# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 03: Neural Basis Expansion (N-BEATS) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In business forecasting (supply chain inventory, retail sales), stakeholders demand **Interpretability**:
- Black-box neural networks (MLPs, LSTMs) make accurate predictions but cannot explain *why*.
- Statistical models (ARIMA, Prophet) are interpretable (explicit trend + seasonality) but struggle with complex non-linear interactions.

In 2020, Oreshkin et al. (Element AI) introduced **N-BEATS (Neural Basis Expansion Analysis for Time Series)**:
- Pure deep learning architecture using **no recurrent units, no convolutions, and no attention**.
- Employs **Backcast and Forecast residual blocks**: each block subtracts its fitted backcast from the input to produce clean residuals for downstream blocks.
- Yields interpretable decomposition into **Trend and Seasonality basis functions**.

---

### 2. Mathematical Foundations: Residual Stacking
For block $l$ receiving input $x_l$:

1. **Fully Connected Expansion:** $\theta_l^b = W_b h_l, \quad \theta_l^f = W_f h_l$
2. **Backcast Output (Reconstructing Past):**
   $$\hat{x}_l = g_l^b(\theta_l^b)$$
3. **Forecast Output (Predicting Future):**
   $$\hat{y}_l = g_l^f(\theta_l^f)$$
4. **Hierarchical Residual Subtraction:**
   $$x_{l+1} = x_l - \hat{x}_l$$
   The next block only models what previous blocks failed to explain!
5. **Global Forecast Accumulation:**
   $$\hat{y}_{\text{total}} = \sum_{l=1}^L \hat{y}_l$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Create Synthetic Historical Demand Signal (Backcast window = 12, Forecast horizon = 4)
backcast_len = 12
forecast_len = 4

# Synthetic signal: Trend + Cyclical wave
t = np.linspace(0, 4 * np.pi, backcast_len + forecast_len)
ground_truth = 2.0 * t + 5.0 * np.sin(t)

x_history = ground_truth[:backcast_len]
y_target = ground_truth[backcast_len:]

print(f"Historical Input (length {backcast_len}):\n", np.round(x_history, 2))
print(f"Future Target to Predict (horizon {forecast_len}):\n", np.round(y_target, 2))

# Step 2: Implement Interpretable N-BEATS Block
class NBeatsBlock:
    def __init__(self, in_len, out_len, hidden_dim=16):
        self.in_len = in_len
        self.out_len = out_len
        # FC stack weights
        self.W1 = np.random.randn(in_len, hidden_dim) * 0.1
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.W_b = np.random.randn(hidden_dim, in_len) * 0.1
        self.W_f = np.random.randn(hidden_dim, out_len) * 0.1

    def forward(self, x):
        # FC feature extraction
        h = np.maximum(0, x @ self.W1)
        h = np.maximum(0, h @ self.W2)
        
        # Backcast (reconstruction of historical input)
        x_hat = h @ self.W_b
        # Forecast (future prediction)
        y_hat = h @ self.W_f
        
        return x_hat, y_hat

# Step 3: Run Multi-Block Residual Architecture (2 Blocks)
block1 = NBeatsBlock(backcast_len, forecast_len)
block2 = NBeatsBlock(backcast_len, forecast_len)

# Block 1 Pass
x_hat_1, y_hat_1 = block1.forward(x_history)
# Residual passed to Block 2
residual_1 = x_history - x_hat_1

# Block 2 Pass on Residuals
x_hat_2, y_hat_2 = block2.forward(residual_1)

# Total Aggregated Forecast
y_final_forecast = y_hat_1 + y_hat_2

print("\n--- N-BEATS Residual Forward Decomposition ---")
print("Block 1 Forecast Component:", np.round(y_hat_1, 2))
print("Block 2 Residual Forecast: ", np.round(y_hat_2, 2))
print("Total Combined Forecast:    ", np.round(y_final_forecast, 2))

### 3. Step-by-Step Code Walkthrough
1. **Backcast Reconstruction:** `x_hat_1` attempts to reconstruct the input history.
2. **Residual Subtraction:** Subtracting the backcast creates a clean error signal (`residual_1`), forcing Block 2 to focus exclusively on unmodeled high-frequency variations.
3. **Additive Ensemble:** The global forecast is the direct sum of all block predictions.

---

### 4. Industry Applications & Trade-offs
- **Electricity & Retail Forecasting:** Won the prestigious M4 and M5 competition benchmarks against traditional statistical methods.
- **Trade-off:** Pure feedforward architecture requires fixed input and output horizons (cannot dynamically produce arbitrary variable-length horizons at inference time).

---

### 5. Top Interview Questions & Answers
**Q1: Why does N-BEATS use backcast reconstruction when our only goal is forecasting future values?**
> **Answer:** The backcast acts as an inductive bias and continuous residual filter. By stripping away what a block already understands about the past, downstream blocks only receive unexplained variance, preventing redundant feature learning and mimicking orthogonal decomposition.

**Q2: What is the difference between Generic and Interpretable N-BEATS?**
> **Answer:** Generic N-BEATS learns unconstrained weights $W_b, W_f$. Interpretable N-BEATS constrains the basis functions $g_l^b, g_l^f$: the Trend block uses polynomial basis functions ($t^p$), and the Seasonality block uses Fourier harmonics (sine and cosine waves).